<a href="https://colab.research.google.com/github/DonChenn/RedditSentimentAnalysis/blob/main/BERTopic_Seeded.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install bertopic
!pip install gensim
from google.colab import drive
drive.mount('/content/drive')
import sys
from bertopic import BERTopic
from umap import UMAP
from sklearn.feature_extraction.text import CountVectorizer

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
%run "/content/drive/MyDrive/Colab Notebooks/CombinedTrainingDataset.ipynb"
%run "/content/drive/MyDrive/Colab Notebooks/StopWords.ipynb"

Using Colab cache for faster access to the 'liberals-vs-conservatives-on-reddit-13000-posts' dataset.
Using Colab cache for faster access to the '1-million-reddit-comments-from-40-subreddits' dataset.
Successfully combined!
Dataset 1 rows: 12854
Dataset 2 ('politics' only) rows: 25000
Total combined rows: 37854


In [10]:
vectorizer_model = CountVectorizer(
    stop_words=my_stop_words,
    ngram_range=(1, 2),
    min_df=10,
    token_pattern=r'(?u)\b[a-zA-Z]{3,}\b'
)

In [11]:
# 25 Targeted Political Seed Topics
# Copy and paste this directly into your Colab notebook

seed_topic_list = [
    # --- ECONOMY & LABOR ---
    # 1. Inflation & Cost of Living
    ["inflation", "prices", "cost", "economy", "fed", "recession", "gas", "groceries",
    "purchasing", "power", "interest", "rates", "supply", "chain", "deficit",
    "spending", "cpi", "consumer", "index", "expensive", "affordability"],
    # 2. Taxes & Wealth Distribution
    ["tax", "taxes", "irs", "wealth", "billionaires", "loophole", "income", "corporate",
    "brackets", "capital", "gains", "revenue", "billionaire", "millionaires", "fair",
    "share", "audit", "deduction", "estates", "rich", "taxation"],
    # 3. Labor Rights & Unions
    ["labor", "unions", "strike", "workers", "wages", "minimum", "pay", "benefits",
    "bargaining", "unionization", "guild", "workplace", "employment", "overtime",
    "safety", "pension", "salaries", "staffing", "contract", "exploitation"],
    # 4. Housing & Real Estate
    ["housing", "rent", "mortgage", "zoning", "homelessness", "affordable", "eviction",
    "landlord", "tenants", "property", "real", "estate", "development", "market",
    "shortage", "equity", "gentrification", "shelter", "units", "inventory"],
    # 5. Student Debt & Education
    ["debt", "student", "loans", "forgiveness", "tuition", "college", "university",
    "degree", "graduates", "interest", "borrowers", "education", "academic",
    "scholarship", "fafsa", "repayment", "cancel", "default", "higher", "learning"],

    # --- HEALTHCARE & PUBLIC HEALTH ---
    # 6. Healthcare & Insurance
    ["insurance", "healthcare", "medicare", "hospital", "coverage", "premiums", "prescription",
    "medicaid", "deductible", "copay", "provider", "doctor", "physician", "clinic", "treatment",
    "patient", "medical", "pharma", "aca", "obamacare"],
    # 7. Abortion & Reproductive Rights
    ["abortion", "roe", "wade", "planned", "parenthood", "fetus", "reproductive", "pregnancy",
    "choice", "prolife", "prochoice", "clinic", "viability", "contraception", "birth",
    "autonomy", "rights", "women", "dobbs", "legal"],
    # 8. Pandemic & Public Health
    ["covid", "pandemic", "vaccine", "mandate", "mask", "fauci", "virus", "health",
    "quarantine", "lockdown", "outbreak", "moderna", "pfizer", "booster", "cdc",
    "transmission", "distancing", "public", "emergency", "variants"],
    # 9. Drug Policy & Decriminalization
    ["drugs", "legalization", "marijuana", "weed", "decriminalization", "addiction", "overdose",
    "cannabis", "opioid", "fentanyl", "narcotics", "treatment", "dispensary", "rehab",
    "recreational", "controlled", "substances", "prison", "sentencing", "possession"],

    # --- CIVIL RIGHTS & JUSTICE ---
    # 10. Gun Control & Second Amendment
    ["gun", "guns", "nra", "amendment", "firearm", "background", "checks", "shooting",
    "violence", "weapons", "rifle", "pistol", "ban", "arms", "safety",
    "massacre", "shooter", "regulation", "carry", "concealed"],
    # 11. LGBTQ+ & Gender Issues
    ["lgbtq", "trans", "transgender", "gay", "marriage", "gender", "rights", "pride",
    "queer", "homosexuality", "bisexual", "pronouns", "identity", "nonbinary", "transition",
    "equality", "discrimination", "hormone", "bathroom", "ban"],
    # 12. Racial Justice & Civil Rights
    ["race", "racism", "systemic", "blm", "black", "white", "diversity", "equity",
    "inclusion", "prejudice", "minority", "supremacy", "segregation", "reparations", "privilege",
    "civil", "rights", "marginalized", "profiling", "justice"],
    # 13. Policing & Law Enforcement
   ["police", "cops", "brutality", "reform", "defund", "officer", "arrest", "accountability",
    "sheriff", "patrol", "shooting", "force", "misconduct", "qualified", "immunity",
    "detective", "badge", "law", "enforcement", "swat"],
    # 14. Criminal Justice & Prison Reform
    ["prison", "justice", "criminal", "reform", "incarceration", "sentence", "inmates",
    "jail", "parole", "probation", "felony", "misdemeanor", "conviction", "recidivism", "rehab",
    "solitary", "warden", "exonerated", "private", "prisons"],

    # --- ENVIRONMENT & INFRASTRUCTURE ---
    # 15. Climate Change & Energy
    ["climate", "warming", "emissions", "carbon", "fossil", "green", "energy", "oil",
    "renewable", "solar", "wind", "methane", "epa", "environment", "pollution",
    "sustainability", "gas", "coal", "grid", "electric"],
    # 16. Infrastructure & Transportation
    ["infrastructure", "roads", "bridges", "transit", "rail", "amtrak", "grid", "transportation",
    "highway", "subway", "trains", "tunnel", "construction", "pipes", "broadband",
    "water", "sewage", "logistics", "electric", "vehicles"],

    # --- GOVERNMENT, ELECTIONS & IMMIGRATION ---
    # 17. Voting Rights & Elections
    ["voting", "voters", "election", "fraud", "ballot", "gerrymandering", "suppression",
    "electoral", "college", "registration", "absentee", "primary", "midterms", "polling", "census",
    "districts", "integrity", "disenfranchisement", "turnout", "early"],
    # 18. Supreme Court & Judiciary
    ["supreme", "court", "scotus", "justices", "judge", "ruling", "bench", "nominee",
    "jurisdiction", "unconstitutional", "precedent", "confirmation", "docket", "circuit", "legal",
    "opinion", "constitution", "judicial", "overturn", "appeal"],
    # 19. Campaign Finance & Corruption
    ["campaign", "finance", "citizens", "united", "pac", "lobbying", "donors", "corruption",
    "superpac", "bribery", "ethics", "lobbyist", "funding", "special", "interests",
    "dark", "money", "disclosure", "clout", "quid"],
    # 20. Executive Power & Investigations
    ["impeachment", "subpoena", "treason", "investigation", "committee", "hearing", "testimony",
    "executive", "privilege", "pardon", "cabinet", "veto", "prosecutor", "contempt", "oversight",
    "scandal", "witness", "perjury", "doj", "order"],
    # 21. Immigration & Border Security
   ["border", "immigration", "immigrants", "wall", "asylum", "ice", "deportation", "migrants",
    "citizenship", "visa", "illegal", "legal", "daca", "patrol", "crossing",
    "refugee", "customs", "undocumented", "naturalization", "h1b"],

    # --- FOREIGN POLICY & TECHNOLOGY ---
    # 22. Russia & Ukraine Conflict
    ["ukraine", "russia", "putin", "zelensky", "nato", "war", "sanctions", "invasion",
    "kremlin", "kyiv", "moscow", "missiles", "intelligence", "offensive", "tanks",
    "aid", "weapons", "ceasefire", "territory", "victory"],
    # 23. China & Global Trade
    ["china", "taiwan", "tariffs", "ccp", "trade", "beijing", "espionage", "manufacturing",
    "semiconductors", "supply", "chain", "exports", "imports", "economy", "sanctions",
    "tensions", "dominance", "competition", "shipping", "xi"],
    # 24. Middle East Relations
    ["israel", "palestine", "gaza", "hamas", "middle", "east", "jewish", "antisemitism",
    "conflict", "iran", "jerusalem", "tel", "aviv", "settlements", "apartheid",
    "hezbollah", "oil", "syria", "treaty", "hostages"],
    # 25. Big Tech, Privacy & Social Media
    ["tech", "privacy", "tiktok", "facebook", "censorship", "algorithm", "data", "monopoly",
    "silicon", "valley", "encryption", "surveillance", "platform", "twitter", "google",
    "antitrust", "regulation", "section", "230", "internet"]
]

In [12]:

docs = combined_data['Combined_Content'].tolist()

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)


topic_model = BERTopic(
    umap_model=umap_model,
    vectorizer_model=vectorizer_model,
    seed_topic_list=seed_topic_list,
    nr_topics=50,
    min_topic_size=40,
    calculate_probabilities=True,
    verbose=True
)
topics, probs = topic_model.fit_transform(docs)
freq = topic_model.get_topic_info()

2026-03-04 22:21:54,535 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1183 [00:00<?, ?it/s]

2026-03-04 22:42:20,811 - BERTopic - Embedding - Completed ✓
2026-03-04 22:42:20,814 - BERTopic - Guided - Find embeddings highly related to seeded topics.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-04 22:42:21,571 - BERTopic - Guided - Completed ✓
2026-03-04 22:42:21,573 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-04 22:43:35,610 - BERTopic - Dimensionality - Completed ✓
2026-03-04 22:43:35,613 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-04 22:44:11,384 - BERTopic - Cluster - Completed ✓
2026-03-04 22:44:11,386 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-03-04 22:44:15,168 - BERTopic - Representation - Completed ✓
2026-03-04 22:44:15,169 - BERTopic - Topic reduction - Reducing number of topics
2026-03-04 22:44:15,244 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-04 22:44:17,610 - BERTopic - Representation - Completed ✓
2026-03-04 22:44:17,618 - BERTopic - Topic reduction - Reduced number of topics from 103 to 50


In [13]:
topic_info = topic_model.get_topic_info()
raw_labels = topic_info['Name'].tolist()
formatted_labels = []

for label in raw_labels:
    parts = label.split('_')
    if parts[0] == "-1":
        clean_label = "Other / Irrelevant / Noise"
    else:
        clean_label = ", ".join(parts[1:])
    formatted_labels.append(clean_label)

for i, label in enumerate(formatted_labels):
    print(f"Index {i}: {label}")

llm_topic_list = formatted_labels

Index 0: Other / Irrelevant / Noise
Index 1: mueller, barr, impeachment, letter
Index 2: ukraine, russia, putin, russian
Index 3: voters, electoral, elections, polls
Index 4: tax, money, taxes, wealth
Index 5: barr, testimony, lied, lying
Index 6: feminist, gender, sex, female
Index 7: black, white, race, racism
Index 8: candidate, primary, candidates, supporters
Index 9: internet, social media, social, account
Index 10: workers, labor, unions, strike
Index 11: prison, jail, prisoners, rights
Index 12: gun, police, guns, shooting
Index 13: fox, news, fox news, report
Index 14: inflation, economy, market, economic
Index 15: insurance, healthcare, health, medicare
Index 16: fat, dude, dick, job
Index 17: capitalism, socialism, socialist, communism
Index 18: funny, thank, okay, gonna
Index 19: abortion, birth, baby, health
Index 20: covid, vaccine, pandemic, medical
Index 21: pelosi, warren, impeachment, nancy
Index 22: graham, lindsey, traitor, talking
Index 23: ban, civil, advocating, v

In [16]:
# Get the raw info
topic_info = topic_model.get_topic_info()

# Calculate the percentage share for each topic
total_docs = topic_info['Count'].sum()
topic_info['Share %'] = (topic_info['Count'] / total_docs) * 100

# Display the top 10 most discussed topics
print(topic_info[['Topic', 'Count', 'Share %', 'Name']])

    Topic  Count    Share %                                               Name
0      -1  15431  40.764516                      -1_report_mueller_barr_public
1       0   2660   7.026998                  0_mueller_barr_impeachment_letter
2       1   1623   4.287526                     1_ukraine_russia_putin_russian
3       2   1487   3.928251                 2_voters_electoral_elections_polls
4       3   1043   2.755323                           3_tax_money_taxes_wealth
5       4    904   2.388123                        4_barr_testimony_lied_lying
6       5    762   2.012997                       5_feminist_gender_sex_female
7       6    681   1.799017                          6_black_white_race_racism
8       7    664   1.754108          7_candidate_primary_candidates_supporters
9       8    653   1.725049             8_internet_social media_social_account
10      9    649   1.714482                      9_workers_labor_unions_strike
11     10    613   1.619380                    10_pr